<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/Task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 5


**Name**: Sohail Essajee (5757504)

This task is a gradio based search for financial news. A deep learning model was implemented to convert the news articles into vectorised embeddings. These embeddings allows us to perform semantic search. This higlights the difference between traditional keyword searching, where semantic searches look for meaning in words.  

Cell outputs are disabled to upload to github.

## Task 5:1

In [ ]:
import pandas as pd
import re

# Load the dataset
df = pd.read_csv('financial_news.csv')

# Define a function to extract and remove the URL
def extract_url(text):

    url_pattern = r'(https?://\S+|www\.\S+)$'

    match = re.search(url_pattern, text)
    if match:
        url = match.group(0)
        clean_text = re.sub(url_pattern, '', text).strip()
        return clean_text, url
    else:
        return text, None

# Apply the function to create the new column
df[['text', 'URL']] = df['text'].apply(lambda x: pd.Series(extract_url(str(x))))


df.to_csv('financial_news_updated.csv', index=False)

print(df.head())

## Task 5:2

In [ ]:
# Install gradio andtransformers
!pip install gradio
!pip install sentence-transformers

import pandas as pd
import gradio as gr
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load your new data
df = pd.read_csv('financial_news_updated.csv')

# 2. Initialize the model and create embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Encoding sentences, please wait...")
corpus_embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

# 3. Define the Search Function
def semantic_search(query):
    query_embedding = model.encode([query])

    # Calculate Cosine Similarity between query and all records
    similarities = cosine_similarity(query_embedding, corpus_embeddings).flatten()

    # Get indices of top 5 matches
    top_indices = similarities.argsort()[-5:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            "Score": round(float(similarities[idx]), 4),
            "Text": df.iloc[idx]['text'],
            "URL": df.iloc[idx]['URL']
        })

    return pd.DataFrame(results)

# 4. Build the Gradio Interface
interface = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(lines=2, placeholder="Enter search terms (e.g., 'regulatory fine')..."),
    outputs=gr.Dataframe(),
    title="Financial News Semantic Search",
    description="Find the top 5 most relevant financial news records using AI embeddings."
)

if __name__ == "__main__":
    interface.launch()